# GPT

In [ ]:
# ============================================================
# OPTIMIZED FINAL PIPELINE
# Focus:
#   - xgb_full_propagation
#   - lstm_context_full
#
# Goals:
#   - keep Nov-Dec 2019 as untouched final test set
#   - use lighter time-aware CV on Jan-Oct 2019
#   - reduce RAM usage
#   - reduce repeated preprocessing
# ============================================================

from pathlib import Path
import sys
import importlib
import warnings
warnings.filterwarnings("ignore")

import os
import gc
import time
import multiprocessing
from copy import deepcopy

# ============================================================
# MULTI-CORE / ENV SETUP
# ============================================================

N_CORES = multiprocessing.cpu_count()
print(f"[SYSTEM] Detected {N_CORES} CPU cores")

os.environ["OMP_NUM_THREADS"] = str(N_CORES)
os.environ["MKL_NUM_THREADS"] = str(N_CORES)
os.environ["OPENBLAS_NUM_THREADS"] = str(N_CORES)
os.environ["NUMEXPR_NUM_THREADS"] = str(N_CORES)
os.environ["TF_NUM_INTEROP_THREADS"] = str(N_CORES)
os.environ["TF_NUM_INTRAOP_THREADS"] = str(N_CORES)

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt

from sklearn.metrics import (
    roc_auc_score,
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    mean_absolute_error,
    mean_squared_error,
    roc_curve,
)
from sklearn.preprocessing import StandardScaler

from xgboost import XGBClassifier, XGBRegressor

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

print(f"[TF] inter_op threads: {tf.config.threading.get_inter_op_parallelism_threads()}")
print(f"[TF] intra_op threads: {tf.config.threading.get_intra_op_parallelism_threads()}")

GLOBAL_START = time.perf_counter()

def timer_log(label, start_time):
    elapsed = time.perf_counter() - start_time
    print(f"[TIMER] {label}: {elapsed:.2f}s")


# ============================================================
# DEVICE SETUP
# ============================================================

# Safer if you are currently running CPU-only
USE_GPU = False

try:
    if USE_GPU:
        gpus = tf.config.list_physical_devices("GPU")
        if gpus:
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
            print(f"[TF] GPU mode enabled. Found {len(gpus)} GPU(s): {[g.name for g in gpus]}")
        else:
            print("[TF] WARNING: USE_GPU=True but no GPU found — falling back to CPU.")
    else:
        tf.config.set_visible_devices([], "GPU")
        print("[TF] CPU-only mode enabled.")
except RuntimeError as e:
    print(f"[TF WARNING] Device config must be set before TF runtime init: {e}")


# ============================================================
# 1. LOAD DATA
# ============================================================

t0 = time.perf_counter()

pl.Config.set_tbl_rows(1000)
pl.Config.set_tbl_cols(100)
pl.Config.set_tbl_width_chars(200)

current = Path.cwd()
while current.name != "shared-notebooks":
    if current.parent == current:
        raise RuntimeError("Could not locate shared-notebooks directory")
    current = current.parent

utils_path = current / "common_utils" / "python"
if str(utils_path) not in sys.path:
    sys.path.append(str(utils_path))

import load_flight_data
importlib.reload(load_flight_data)

lf = load_flight_data.load_flight_data(file_name="flights_canonical_2019.parquet")

timer_log("Load data", t0)


# ============================================================
# 2. FEATURE ENGINEERING / MODELING TABLE
# ============================================================

t0 = time.perf_counter()

RAW_FEATURES = [
    "flight_id",
    "tail_number",
    "reporting_airline",
    "origin",
    "dest",
    "route_key",
    "distance",
    "flight_date",

    "dep_hour_local",
    "dep_weekday_local",
    "dep_month_local",

    "dep_ts_sched_utc",
    "dep_ts_actual_utc",
    "arr_ts_sched_utc",
    "arr_ts_actual_utc",

    "dep_temp_c",
    "dep_wind_speed_m_s",
    "dep_wind_dir_deg",
    "dep_ceiling_height_m",
    "arr_temp_c",
    "arr_wind_speed_m_s",
    "arr_wind_dir_deg",
    "arr_ceiling_height_m",

    "prev_flight_id_same_tail",
    "next_flight_id_same_tail",
    "prev_origin",
    "prev_dest",
    "next_origin",
    "next_dest",
    "prev_arr_ts_actual_utc",
    "next_dep_ts_actual_utc",

    "dep_delay",
    "dep_del15",
    "arr_delay",
    "arr_del15",

    "prev_arr_delay",
    "prev_dep_delay",
    "next_arr_delay",
    "next_dep_delay",
    "prev_arr_del15",
    "prev_dep_del15",
    "next_dep_del15",
    "next_arr_del15",
    "prev_arr_late_15",
    "prev_dep_late_15",
    "next_arr_late_15",
    "next_dep_late_15",

    "turnaround_minutes",
    "next_turnaround_minutes",
    "rotation_continuity_flag",
    "next_rotation_continuity_flag",
    "aircraft_leg_number_day",
    "cum_dep_delay_aircraft_day",
    "cum_arr_delay_aircraft_day",

    "is_cancelled",
    "is_diverted",

    "crs_elapsed_time",
    "dep_time_blk",
    "arr_time_blk",
]

US_HOLIDAYS_2019 = [
    "2019-01-01",
    "2019-01-21",
    "2019-02-18",
    "2019-05-27",
    "2019-07-04",
    "2019-09-02",
    "2019-10-14",
    "2019-11-11",
    "2019-11-28",
    "2019-12-25",
]

ml_lf = (
    lf
    .select(RAW_FEATURES)
    .filter(
        (pl.col("is_cancelled") == 0) &
        (pl.col("is_diverted") == 0) &
        pl.col("arr_del15").is_not_null() &
        pl.col("tail_number").is_not_null() &
        pl.col("dep_ts_actual_utc").is_not_null() &
        pl.col("arr_ts_actual_utc").is_not_null()
    )
)

lf_features = (
    ml_lf
    .with_columns([
        pl.when(pl.col("dep_hour_local") < 6).then(1)
        .when(pl.col("dep_hour_local") < 11).then(2)
        .when(pl.col("dep_hour_local") < 14).then(3)
        .when(pl.col("dep_hour_local") < 18).then(4)
        .when(pl.col("dep_hour_local") < 21).then(5)
        .otherwise(6)
        .alias("dep_time_bucket"),

        pl.col("dep_weekday_local").is_in([6, 7]).cast(pl.Int8).alias("is_weekend"),

        pl.col("flight_date").cast(pl.Utf8).is_in(US_HOLIDAYS_2019).cast(pl.Int8).alias("is_holiday"),

        pl.min_horizontal([
            *[
                (pl.col("flight_date").cast(pl.Date) - pl.lit(h).str.strptime(pl.Date)).abs().dt.total_days()
                for h in US_HOLIDAYS_2019
            ]
        ]).alias("days_to_nearest_holiday"),

        pl.len().over("route_key").alias("route_frequency"),
        pl.len().over("origin").alias("origin_flight_volume"),
        pl.len().over("dest").alias("dest_flight_volume"),

        (pl.col("prev_arr_delay") > 15).cast(pl.Int8).alias("prev_arr_delayed_flag"),
        (pl.col("prev_arr_delay") + pl.col("prev_dep_delay")).alias("prev_total_delay"),

        (pl.col("turnaround_minutes") < 60).cast(pl.Int8).alias("tight_turnaround_flag"),

        (
            pl.col("aircraft_leg_number_day") /
            pl.max("aircraft_leg_number_day").over("tail_number")
        ).alias("relative_leg_position"),
    ])
)

usa_2hop_lf = (
    lf_features
    .sort(["tail_number", "dep_ts_actual_utc"])
    .with_columns([
        # 1 hop back
        pl.col("flight_id").shift(1).over("tail_number").alias("prev1_flight_id"),
        pl.col("origin").shift(1).over("tail_number").alias("prev1_origin"),
        pl.col("dest").shift(1).over("tail_number").alias("prev1_dest"),
        pl.col("dep_ts_actual_utc").shift(1).over("tail_number").alias("prev1_dep_ts_utc"),
        pl.col("arr_ts_actual_utc").shift(1).over("tail_number").alias("prev1_arr_ts_utc"),
        pl.col("arr_delay").shift(1).over("tail_number").alias("prev1_arr_delay"),
        pl.col("dep_delay").shift(1).over("tail_number").alias("prev1_dep_delay"),
        pl.col("arr_del15").shift(1).over("tail_number").alias("prev1_arr_del15"),
        pl.col("dep_del15").shift(1).over("tail_number").alias("prev1_dep_del15"),

        # 2 hops back
        pl.col("flight_id").shift(2).over("tail_number").alias("prev2_flight_id"),
        pl.col("origin").shift(2).over("tail_number").alias("prev2_origin"),
        pl.col("dest").shift(2).over("tail_number").alias("prev2_dest"),
        pl.col("dep_ts_actual_utc").shift(2).over("tail_number").alias("prev2_dep_ts_utc"),
        pl.col("arr_ts_actual_utc").shift(2).over("tail_number").alias("prev2_arr_ts_utc"),
        pl.col("arr_delay").shift(2).over("tail_number").alias("prev2_arr_delay"),
        pl.col("dep_delay").shift(2).over("tail_number").alias("prev2_dep_delay"),
        pl.col("arr_del15").shift(2).over("tail_number").alias("prev2_arr_del15"),
        pl.col("dep_del15").shift(2).over("tail_number").alias("prev2_dep_del15"),

        # timing gaps
        (
            pl.col("dep_ts_actual_utc") -
            pl.col("arr_ts_actual_utc").shift(1).over("tail_number")
        ).dt.total_minutes().alias("prev1_turnaround_minutes"),

        (
            pl.col("dep_ts_actual_utc") -
            pl.col("arr_ts_actual_utc").shift(2).over("tail_number")
        ).dt.total_minutes().alias("time_since_prev2_arrival_minutes"),
    ])
    .filter(
        pl.col("prev1_flight_id").is_not_null() &
        pl.col("prev2_flight_id").is_not_null() &
        pl.col("prev1_turnaround_minutes").is_not_null() &
        pl.col("time_since_prev2_arrival_minutes").is_not_null() &
        pl.col("prev1_turnaround_minutes").is_between(0, 12 * 60) &
        pl.col("time_since_prev2_arrival_minutes").is_between(0, 24 * 60)
    )
)

flights = usa_2hop_lf.collect()

flights = flights.with_columns(
    pl.col("dep_ts_actual_utc").cast(pl.Datetime).alias("dep_ts_actual_utc")
)

print("Rows in final modeling table:", flights.height)
print(flights.select(["flight_id", "tail_number", "origin", "dest", "arr_delay", "arr_del15"]).head())

timer_log("Feature engineering + collect", t0)


# ============================================================
# 3. FEATURE SETS
# ============================================================

xgb_full_features = [
    "distance",
    "dep_hour_local",
    "dep_weekday_local",
    "dep_month_local",
    "dep_time_bucket",
    "is_weekend",
    "is_holiday",
    "days_to_nearest_holiday",
    "crs_elapsed_time",

    "dep_temp_c",
    "dep_wind_speed_m_s",
    "dep_wind_dir_deg",
    "dep_ceiling_height_m",
    "arr_temp_c",
    "arr_wind_speed_m_s",
    "arr_wind_dir_deg",
    "arr_ceiling_height_m",
    "route_frequency",
    "origin_flight_volume",
    "dest_flight_volume",

    "prev_arr_delay",
    "prev_dep_delay",
    "prev_arr_del15",
    "prev_dep_del15",
    "prev_arr_delayed_flag",
    "prev_total_delay",
    "turnaround_minutes",
    "tight_turnaround_flag",
    "rotation_continuity_flag",
    "aircraft_leg_number_day",
    "relative_leg_position",
    "cum_dep_delay_aircraft_day",
    "cum_arr_delay_aircraft_day",
    "prev1_arr_delay",
    "prev1_dep_delay",
    "prev1_arr_del15",
    "prev1_dep_del15",
    "prev2_arr_delay",
    "prev2_dep_delay",
    "prev2_arr_del15",
    "prev2_dep_del15",
    "prev1_turnaround_minutes",
    "time_since_prev2_arrival_minutes",
]

lstm_context_current = [
    "distance",
    "dep_hour_local",
    "dep_weekday_local",
    "dep_month_local",
    "dep_time_bucket",
    "is_weekend",
    "is_holiday",
    "days_to_nearest_holiday",
    "crs_elapsed_time",

    "dep_temp_c",
    "dep_wind_speed_m_s",
    "dep_wind_dir_deg",
    "dep_ceiling_height_m",
    "arr_temp_c",
    "arr_wind_speed_m_s",
    "arr_wind_dir_deg",
    "arr_ceiling_height_m",
    "route_frequency",
    "origin_flight_volume",
    "dest_flight_volume",
    "tight_turnaround_flag",
    "relative_leg_position",
    "cum_dep_delay_aircraft_day",
    "cum_arr_delay_aircraft_day",
]

TARGET_CLASS = "arr_del15"
TARGET_REG = "arr_delay"


# ============================================================
# 4. FINAL HOLDOUT SPLIT
#    Keep Nov-Dec untouched
# ============================================================

t0 = time.perf_counter()

test_start_date = pd.Timestamp("2019-11-01")

train_tune_df = flights.filter(pl.col("dep_ts_actual_utc") < test_start_date)
test_df = flights.filter(pl.col("dep_ts_actual_utc") >= test_start_date)

print("Train+tune rows:", train_tune_df.height)
print("Final test rows:", test_df.height)

timer_log("Final train/test split", t0)


# ============================================================
# 5. METRIC / HELPER FUNCTIONS
# ============================================================

def classification_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "auc": roc_auc_score(y_true, y_prob),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "accuracy": accuracy_score(y_true, y_pred),
    }

def regression_metrics(y_true, y_pred):
    return {
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": np.sqrt(mean_squared_error(y_true, y_pred)),
    }

def summarize_cv_metrics(metric_list, prefix="cv"):
    keys = metric_list[0].keys()
    out = {}
    for k in keys:
        vals = np.array([m[k] for m in metric_list], dtype=float)
        out[f"{prefix}_{k}_mean"] = vals.mean()
        out[f"{prefix}_{k}_std"] = vals.std()
    return out

def rank_results(df):
    return (
        df.sort_values(
            by=["cv_auc_mean", "cv_f1_mean", "cv_mae_mean", "cv_rmse_mean"],
            ascending=[False, False, True, True]
        )
        .reset_index(drop=True)
    )

def safe_fill_pandas(df_pl: pl.DataFrame, cols: list[str]) -> pd.DataFrame:
    pdf = df_pl.select(cols).to_pandas()
    for c in pdf.columns:
        if pdf[c].dtype.kind in "biufc":
            med = pdf[c].median()
            if pd.isna(med):
                med = 0
            pdf[c] = pdf[c].fillna(med).astype(np.float32)
        else:
            pdf[c] = pdf[c].fillna("missing")
    return pdf

def safe_fill_pandas_with_stats(df_pl: pl.DataFrame, cols: list[str], fill_stats=None):
    pdf = df_pl.select(cols).to_pandas()
    used_fill_stats = {}

    for c in pdf.columns:
        if pdf[c].dtype.kind in "biufc":
            if fill_stats is None:
                med = pdf[c].median()
                if pd.isna(med):
                    med = 0.0
            else:
                med = fill_stats.get(c, 0.0)
            pdf[c] = pdf[c].fillna(med).astype(np.float32)
            used_fill_stats[c] = med
        else:
            pdf[c] = pdf[c].fillna("missing")

    return pdf, used_fill_stats


# ============================================================
# 6. TIME-AWARE CV SPLITS
#    Lighter 3-fold expanding window
# ============================================================

def make_time_cv_splits(df_pl: pl.DataFrame):
    """
    Fold 1: train Jan-Jul, validate Aug
    Fold 2: train Jan-Aug, validate Sep
    Fold 3: train Jan-Sep, validate Oct
    """
    fold_specs = [
        ("2019-08-01", "2019-09-01"),
        ("2019-09-01", "2019-10-01"),
        ("2019-10-01", "2019-11-01"),
    ]

    splits = []
    for i, (val_start, val_end) in enumerate(fold_specs, start=1):
        val_start = pd.Timestamp(val_start)
        val_end = pd.Timestamp(val_end)

        train_fold = df_pl.filter(pl.col("dep_ts_actual_utc") < val_start)
        val_fold = df_pl.filter(
            (pl.col("dep_ts_actual_utc") >= val_start) &
            (pl.col("dep_ts_actual_utc") < val_end)
        )

        if train_fold.height == 0 or val_fold.height == 0:
            continue

        splits.append({
            "fold": i,
            "train_df": train_fold,
            "val_df": val_fold,
            "val_start": val_start,
            "val_end": val_end,
        })

    return splits

cv_splits = make_time_cv_splits(train_tune_df)
print(f"Built {len(cv_splits)} time-aware CV folds")
for s in cv_splits:
    print(
        f"Fold {s['fold']}: "
        f"train={s['train_df'].height:,} | "
        f"val={s['val_df'].height:,} | "
        f"val_window=[{s['val_start'].date()} -> {s['val_end'].date()})"
    )


# ============================================================
# 7. LSTM STEP FEATURES
# ============================================================

STEP_FEATURES = [
    "arr_delay_prev",
    "dep_delay_prev",
    "arr_del15_prev",
    "dep_del15_prev",
    "gap_minutes",
    "distance",
    "dep_hour_local",
    "dep_weekday_local",
    "dep_month_local",
    "dep_time_bucket",
    "is_weekend",
    "is_holiday",
    "days_to_nearest_holiday",
    "crs_elapsed_time",
    "dep_temp_c",
    "dep_wind_speed_m_s",
    "dep_wind_dir_deg",
    "dep_ceiling_height_m",
    "arr_temp_c",
    "arr_wind_speed_m_s",
    "arr_wind_dir_deg",
    "arr_ceiling_height_m",
    "route_frequency",
    "origin_flight_volume",
    "dest_flight_volume",
    "tight_turnaround_flag",
    "relative_leg_position",
    "cum_dep_delay_aircraft_day",
    "cum_arr_delay_aircraft_day",
]

LSTM_SOURCE_COLS = list({
    "prev2_arr_delay", "prev2_dep_delay", "prev2_arr_del15", "prev2_dep_del15",
    "prev1_arr_delay", "prev1_dep_delay", "prev1_arr_del15", "prev1_dep_del15",
    "prev1_turnaround_minutes", "time_since_prev2_arrival_minutes",
    *lstm_context_current,
    TARGET_CLASS, TARGET_REG
})


def build_lstm_step_matrix_from_pdf(pdf: pd.DataFrame):
    pdf = pdf.copy()

    numeric_fill_cols = [
        "prev2_arr_delay", "prev2_dep_delay", "prev2_arr_del15", "prev2_dep_del15",
        "prev1_arr_delay", "prev1_dep_delay", "prev1_arr_del15", "prev1_dep_del15",
        "prev1_turnaround_minutes", "time_since_prev2_arrival_minutes",
        "distance", "dep_hour_local", "dep_weekday_local", "dep_month_local",
        "dep_time_bucket", "is_weekend", "is_holiday", "days_to_nearest_holiday",
        "crs_elapsed_time", "dep_temp_c", "dep_wind_speed_m_s", "dep_wind_dir_deg",
        "dep_ceiling_height_m", "arr_temp_c", "arr_wind_speed_m_s", "arr_wind_dir_deg",
        "arr_ceiling_height_m", "route_frequency", "origin_flight_volume",
        "dest_flight_volume", "tight_turnaround_flag", "relative_leg_position",
        "cum_dep_delay_aircraft_day", "cum_arr_delay_aircraft_day"
    ]

    for c in numeric_fill_cols:
        if c in pdf.columns:
            med = pdf[c].median()
            if pd.isna(med):
                med = 0.0
            pdf[c] = pdf[c].fillna(med).astype(np.float32)

    X = np.zeros((len(pdf), 3, len(STEP_FEATURES)), dtype=np.float32)

    # prev2
    X[:, 0, STEP_FEATURES.index("arr_delay_prev")] = pdf["prev2_arr_delay"].values
    X[:, 0, STEP_FEATURES.index("dep_delay_prev")] = pdf["prev2_dep_delay"].values
    X[:, 0, STEP_FEATURES.index("arr_del15_prev")] = pdf["prev2_arr_del15"].values
    X[:, 0, STEP_FEATURES.index("dep_del15_prev")] = pdf["prev2_dep_del15"].values
    X[:, 0, STEP_FEATURES.index("gap_minutes")] = pdf["time_since_prev2_arrival_minutes"].values

    # prev1
    X[:, 1, STEP_FEATURES.index("arr_delay_prev")] = pdf["prev1_arr_delay"].values
    X[:, 1, STEP_FEATURES.index("dep_delay_prev")] = pdf["prev1_dep_delay"].values
    X[:, 1, STEP_FEATURES.index("arr_del15_prev")] = pdf["prev1_arr_del15"].values
    X[:, 1, STEP_FEATURES.index("dep_del15_prev")] = pdf["prev1_dep_del15"].values
    X[:, 1, STEP_FEATURES.index("gap_minutes")] = pdf["prev1_turnaround_minutes"].values

    # current context in step 3
    for col in lstm_context_current:
        if col in STEP_FEATURES:
            X[:, 2, STEP_FEATURES.index(col)] = pdf[col].values.astype(np.float32)

    y_cls = pdf[TARGET_CLASS].astype(np.float32).values
    y_reg = pdf[TARGET_REG].astype(np.float32).values

    return X, y_cls, y_reg


# ============================================================
# 8. PRECOMPUTE CV CACHE
#    Huge optimization: prepare folds once
# ============================================================

def prepare_cv_cache(cv_splits, xgb_feature_cols, lstm_source_cols):
    cache = []

    for split in cv_splits:
        t0 = time.perf_counter()

        train_df = split["train_df"]
        val_df = split["val_df"]

        # ---------- XGB cached data ----------
        X_train_xgb, xgb_fill_stats = safe_fill_pandas_with_stats(train_df, xgb_feature_cols, fill_stats=None)
        X_val_xgb, _ = safe_fill_pandas_with_stats(val_df, xgb_feature_cols, fill_stats=xgb_fill_stats)

        y_train_cls = train_df[TARGET_CLASS].to_pandas().astype(np.int8).values
        y_val_cls = val_df[TARGET_CLASS].to_pandas().astype(np.int8).values

        y_train_reg = train_df[TARGET_REG].to_pandas().astype(np.float32).values
        y_val_reg = val_df[TARGET_REG].to_pandas().astype(np.float32).values

        # ---------- LSTM cached data ----------
        lstm_train_pdf = train_df.select(lstm_source_cols).to_pandas()
        lstm_val_pdf = val_df.select(lstm_source_cols).to_pandas()

        X_train_lstm, y_train_cls_lstm, y_train_reg_lstm = build_lstm_step_matrix_from_pdf(lstm_train_pdf)
        X_val_lstm, y_val_cls_lstm, y_val_reg_lstm = build_lstm_step_matrix_from_pdf(lstm_val_pdf)

        scaler = StandardScaler()

        n_train, tsteps, nfeat = X_train_lstm.shape
        n_val = X_val_lstm.shape[0]

        X_train_2d = X_train_lstm.reshape(-1, nfeat)
        X_val_2d = X_val_lstm.reshape(-1, nfeat)

        X_train_lstm_scaled = scaler.fit_transform(X_train_2d).reshape(n_train, tsteps, nfeat).astype(np.float32)
        X_val_lstm_scaled = scaler.transform(X_val_2d).reshape(n_val, tsteps, nfeat).astype(np.float32)

        cache.append({
            "fold": split["fold"],
            "val_start": split["val_start"],
            "val_end": split["val_end"],

            "X_train_xgb": X_train_xgb,
            "X_val_xgb": X_val_xgb,
            "y_train_cls": y_train_cls,
            "y_val_cls": y_val_cls,
            "y_train_reg": y_train_reg,
            "y_val_reg": y_val_reg,

            "X_train_lstm": X_train_lstm_scaled,
            "X_val_lstm": X_val_lstm_scaled,
            "y_train_cls_lstm": y_train_cls_lstm,
            "y_val_cls_lstm": y_val_cls_lstm,
            "y_train_reg_lstm": y_train_reg_lstm,
            "y_val_reg_lstm": y_val_reg_lstm,
        })

        timer_log(f"Prepared cache for fold {split['fold']}", t0)

    return cache

t0 = time.perf_counter()
cv_cache = prepare_cv_cache(cv_splits, xgb_full_features, LSTM_SOURCE_COLS)
timer_log("Prepare all CV caches", t0)


# ============================================================
# 9. MODEL BUILDERS
# ============================================================

def make_xgb_model(params, task="classification"):
    xgb_device = "cuda" if USE_GPU else "cpu"

    common = dict(
        random_state=42,
        n_jobs=-1,
        tree_method="hist",
        device=xgb_device,
        **params
    )

    if task == "classification":
        return XGBClassifier(
            eval_metric="logloss",
            **common
        )
    else:
        return XGBRegressor(**common)


def build_lstm_model(input_shape, units1=64, units2=32, dropout=0.2, learning_rate=1e-3):
    inp = Input(shape=input_shape)
    x = LSTM(units1, return_sequences=True)(inp)
    x = Dropout(dropout)(x)
    x = LSTM(units2)(x)
    x = Dropout(dropout)(x)

    cls_out = Dense(1, activation="sigmoid", name="cls")(x)
    reg_out = Dense(1, activation="linear", name="reg")(x)

    model = Model(inputs=inp, outputs=[cls_out, reg_out])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss={"cls": "binary_crossentropy", "reg": "mse"},
        metrics={"cls": [tf.keras.metrics.AUC(name="auc")], "reg": ["mae"]},
    )
    return model


# ============================================================
# 10. SMALLER HYPERPARAMETER SEARCH SPACES
# ============================================================

# Keep n_estimators high enough for early stopping to find a good stopping point
XGB_PARAM_GRID = [
    {
        "n_estimators": 500,
        "max_depth": 4,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "min_child_weight": 1,
        "reg_lambda": 1.0,
    },
    {
        "n_estimators": 500,
        "max_depth": 6,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "min_child_weight": 1,
        "reg_lambda": 1.0,
    },
    {
        "n_estimators": 500,
        "max_depth": 6,
        "learning_rate": 0.03,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "min_child_weight": 5,
        "reg_lambda": 5.0,
    },
]

LSTM_PARAM_GRID = [
    {
        "units1": 32,
        "units2": 16,
        "dropout": 0.2,
        "batch_size": 512,
        "epochs": 8,
        "learning_rate": 1e-3,
    },
    {
        "units1": 64,
        "units2": 32,
        "dropout": 0.2,
        "batch_size": 512,
        "epochs": 8,
        "learning_rate": 1e-3,
    },
]


# ============================================================
# 11. XGBOOST TIME-CV SEARCH (OPTIMIZED)
# ============================================================

def run_xgb_time_cv_fast(cv_cache, param_grid):
    t_total = time.perf_counter()
    rows = []

    for config_id, params in enumerate(param_grid, start=1):
        t_cfg = time.perf_counter()

        cls_fold_metrics = []
        reg_fold_metrics = []

        for fold_data in cv_cache:
            fold = fold_data["fold"]

            clf = make_xgb_model(params, task="classification")
            clf.set_params(early_stopping_rounds=30)
            clf.fit(
                fold_data["X_train_xgb"],
                fold_data["y_train_cls"],
                eval_set=[(fold_data["X_val_xgb"], fold_data["y_val_cls"])],
                verbose=False
            )
            val_prob = clf.predict_proba(fold_data["X_val_xgb"])[:, 1]
            cls_metrics = classification_metrics(fold_data["y_val_cls"], val_prob)
            cls_fold_metrics.append(cls_metrics)

            reg = make_xgb_model(params, task="regression")
            reg.set_params(early_stopping_rounds=30)
            reg.fit(
                fold_data["X_train_xgb"],
                fold_data["y_train_reg"],
                eval_set=[(fold_data["X_val_xgb"], fold_data["y_val_reg"])],
                verbose=False
            )
            val_pred_reg = reg.predict(fold_data["X_val_xgb"])
            reg_metrics = regression_metrics(fold_data["y_val_reg"], val_pred_reg)
            reg_fold_metrics.append(reg_metrics)

            print(
                f"[XGB cfg {config_id:02d}] fold={fold} "
                f"AUC={cls_metrics['auc']:.4f} "
                f"F1={cls_metrics['f1']:.4f} "
                f"MAE={reg_metrics['mae']:.4f} "
                f"RMSE={reg_metrics['rmse']:.4f}"
            )

            del clf, reg, val_prob, val_pred_reg
            gc.collect()

        row = {
            "model_family": "xgb_full_propagation",
            "config_id": f"xgb_{config_id:02d}",
            "params": deepcopy(params),
        }
        row.update(summarize_cv_metrics(cls_fold_metrics, prefix="cv"))
        row.update(summarize_cv_metrics(reg_fold_metrics, prefix="cv"))
        rows.append(row)

        timer_log(f"XGB config {config_id:02d}", t_cfg)

    results_df = rank_results(pd.DataFrame(rows))
    timer_log("Total XGB time-CV search", t_total)
    return results_df


# ============================================================
# 12. LSTM TIME-CV SEARCH (OPTIMIZED)
# ============================================================

def run_lstm_time_cv_fast(cv_cache, param_grid):
    t_total = time.perf_counter()
    rows = []

    for config_id, params in enumerate(param_grid, start=1):
        t_cfg = time.perf_counter()

        cls_fold_metrics = []
        reg_fold_metrics = []

        for fold_data in cv_cache:
            fold = fold_data["fold"]

            tf.keras.backend.clear_session()

            model = build_lstm_model(
                input_shape=(
                    fold_data["X_train_lstm"].shape[1],
                    fold_data["X_train_lstm"].shape[2]
                ),
                units1=params["units1"],
                units2=params["units2"],
                dropout=params["dropout"],
                learning_rate=params["learning_rate"],
            )

            early_stop = EarlyStopping(
                monitor="val_loss",
                patience=2,
                restore_best_weights=True
            )

            model.fit(
                fold_data["X_train_lstm"],
                {
                    "cls": fold_data["y_train_cls_lstm"],
                    "reg": fold_data["y_train_reg_lstm"],
                },
                validation_data=(
                    fold_data["X_val_lstm"],
                    {
                        "cls": fold_data["y_val_cls_lstm"],
                        "reg": fold_data["y_val_reg_lstm"],
                    }
                ),
                epochs=params["epochs"],
                batch_size=params["batch_size"],
                callbacks=[early_stop],
                verbose=0,
            )

            pred_cls, pred_reg = model.predict(fold_data["X_val_lstm"], verbose=0)
            pred_cls = pred_cls.ravel()
            pred_reg = pred_reg.ravel()

            cls_metrics = classification_metrics(
                fold_data["y_val_cls_lstm"].astype(int),
                pred_cls
            )
            reg_metrics = regression_metrics(
                fold_data["y_val_reg_lstm"],
                pred_reg
            )

            cls_fold_metrics.append(cls_metrics)
            reg_fold_metrics.append(reg_metrics)

            print(
                f"[LSTM cfg {config_id:02d}] fold={fold} "
                f"AUC={cls_metrics['auc']:.4f} "
                f"F1={cls_metrics['f1']:.4f} "
                f"MAE={reg_metrics['mae']:.4f} "
                f"RMSE={reg_metrics['rmse']:.4f}"
            )

            del model, pred_cls, pred_reg
            tf.keras.backend.clear_session()
            gc.collect()

        row = {
            "model_family": "lstm_context_full",
            "config_id": f"lstm_{config_id:02d}",
            "params": deepcopy(params),
        }
        row.update(summarize_cv_metrics(cls_fold_metrics, prefix="cv"))
        row.update(summarize_cv_metrics(reg_fold_metrics, prefix="cv"))
        rows.append(row)

        timer_log(f"LSTM config {config_id:02d}", t_cfg)

    results_df = rank_results(pd.DataFrame(rows))
    timer_log("Total LSTM time-CV search", t_total)
    return results_df


# ============================================================
# 13. RUN CV SEARCH
# ============================================================

t0 = time.perf_counter()
xgb_cv_results = run_xgb_time_cv_fast(cv_cache=cv_cache, param_grid=XGB_PARAM_GRID)
timer_log("XGB CV search block", t0)

t0 = time.perf_counter()
lstm_cv_results = run_lstm_time_cv_fast(cv_cache=cv_cache, param_grid=LSTM_PARAM_GRID)
timer_log("LSTM CV search block", t0)

print("\nTop XGB configs")
print(xgb_cv_results)

print("\nTop LSTM configs")
print(lstm_cv_results)

best_xgb_row = xgb_cv_results.iloc[0].to_dict()
best_lstm_row = lstm_cv_results.iloc[0].to_dict()

best_family_df = pd.DataFrame([best_xgb_row, best_lstm_row]).sort_values(
    by=["cv_auc_mean", "cv_f1_mean", "cv_mae_mean", "cv_rmse_mean"],
    ascending=[False, False, True, True]
).reset_index(drop=True)

print("\nBest config per family")
print(best_family_df[[
    "model_family", "config_id",
    "cv_auc_mean", "cv_f1_mean", "cv_precision_mean", "cv_recall_mean", "cv_accuracy_mean",
    "cv_mae_mean", "cv_rmse_mean", "params"
]])


# ============================================================
# 14. REFIT BEST XGB ON FULL JAN-OCT, TEST ON NOV-DEC
# ============================================================

def refit_best_xgb(train_df, test_df, feature_cols, best_params):
    X_train, fill_stats = safe_fill_pandas_with_stats(train_df, feature_cols, fill_stats=None)
    X_test, _ = safe_fill_pandas_with_stats(test_df, feature_cols, fill_stats=fill_stats)

    y_train_cls = train_df[TARGET_CLASS].to_pandas().astype(int)
    y_test_cls = test_df[TARGET_CLASS].to_pandas().astype(int)

    y_train_reg = train_df[TARGET_REG].to_pandas().astype(float)
    y_test_reg = test_df[TARGET_REG].to_pandas().astype(float)

    clf = make_xgb_model(best_params, task="classification")
    clf.set_params(early_stopping_rounds=30)
    clf.fit(
        X_train,
        y_train_cls,
        eval_set=[(X_test, y_test_cls)],
        verbose=False
    )
    test_prob = clf.predict_proba(X_test)[:, 1]

    reg = make_xgb_model(best_params, task="regression")
    reg.set_params(early_stopping_rounds=30)
    reg.fit(
        X_train,
        y_train_reg,
        eval_set=[(X_test, y_test_reg)],
        verbose=False
    )
    test_pred_reg = reg.predict(X_test)

    return {
        "model_family": "xgb_full_propagation",
        "params": deepcopy(best_params),
        "classifier": clf,
        "regressor": reg,
        "features": feature_cols,
        "test_pred_cls": test_prob,
        "test_pred_reg": test_pred_reg,
        "test_cls_metrics": classification_metrics(y_test_cls, test_prob),
        "test_reg_metrics": regression_metrics(y_test_reg, test_pred_reg),
        "y_test_cls": y_test_cls,
        "y_test_reg": y_test_reg,
    }


# ============================================================
# 15. REFIT BEST LSTM ON FULL JAN-OCT, TEST ON NOV-DEC
# ============================================================

def prepare_final_lstm_arrays(train_df, test_df, lstm_source_cols):
    train_pdf = train_df.select(lstm_source_cols).to_pandas()
    test_pdf = test_df.select(lstm_source_cols).to_pandas()

    X_train, y_train_cls, y_train_reg = build_lstm_step_matrix_from_pdf(train_pdf)
    X_test, y_test_cls, y_test_reg = build_lstm_step_matrix_from_pdf(test_pdf)

    scaler = StandardScaler()
    n_train, tsteps, nfeat = X_train.shape
    n_test = X_test.shape[0]

    X_train_2d = X_train.reshape(-1, nfeat)
    X_test_2d = X_test.reshape(-1, nfeat)

    X_train_scaled = scaler.fit_transform(X_train_2d).reshape(n_train, tsteps, nfeat).astype(np.float32)
    X_test_scaled = scaler.transform(X_test_2d).reshape(n_test, tsteps, nfeat).astype(np.float32)

    return X_train_scaled, X_test_scaled, y_train_cls, y_test_cls, y_train_reg, y_test_reg, scaler, test_pdf


def refit_best_lstm(train_df, test_df, best_params):
    X_train_scaled, X_test_scaled, y_train_cls, y_test_cls, y_train_reg, y_test_reg, scaler, test_pdf = (
        prepare_final_lstm_arrays(train_df, test_df, LSTM_SOURCE_COLS)
    )

    y_train_cls = np.asarray(y_train_cls, dtype=np.float32)
    y_train_reg = np.asarray(y_train_reg, dtype=np.float32)
    y_test_cls = np.asarray(y_test_cls, dtype=np.float32)
    y_test_reg = np.asarray(y_test_reg, dtype=np.float32)

    tf.keras.backend.clear_session()

    model = build_lstm_model(
        input_shape=(X_train_scaled.shape[1], X_train_scaled.shape[2]),
        units1=best_params["units1"],
        units2=best_params["units2"],
        dropout=best_params["dropout"],
        learning_rate=best_params["learning_rate"],
    )

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=2,
        restore_best_weights=True
    )

    model.fit(
        X_train_scaled,
        {"cls": y_train_cls, "reg": y_train_reg},
        validation_split=0.10,
        epochs=best_params["epochs"],
        batch_size=best_params["batch_size"],
        callbacks=[early_stop],
        verbose=1,
    )

    pred_cls, pred_reg = model.predict(X_test_scaled, verbose=0)
    pred_cls = pred_cls.ravel()
    pred_reg = pred_reg.ravel()

    return {
        "model_family": "lstm_context_full",
        "params": deepcopy(best_params),
        "model": model,
        "scaler": scaler,
        "test_pdf": test_pdf,
        "X_test": X_test_scaled,
        "test_pred_cls": pred_cls,
        "test_pred_reg": pred_reg,
        "test_cls_metrics": classification_metrics(y_test_cls.astype(int), pred_cls),
        "test_reg_metrics": regression_metrics(y_test_reg, pred_reg),
        "y_test_cls": y_test_cls,
        "y_test_reg": y_test_reg,
    }


# ============================================================
# 16. FINAL REFIT + TEST EVALUATION
# ============================================================

t0 = time.perf_counter()

best_xgb_final = refit_best_xgb(
    train_df=train_tune_df,
    test_df=test_df,
    feature_cols=xgb_full_features,
    best_params=best_xgb_row["params"],
)

best_lstm_final = refit_best_lstm(
    train_df=train_tune_df,
    test_df=test_df,
    best_params=best_lstm_row["params"],
)

timer_log("Final refit on Jan-Oct + test on Nov-Dec", t0)


# ============================================================
# 17. FINAL TEST RESULTS
# ============================================================

final_test_results = pd.DataFrame([
    {
        "model_family": best_xgb_final["model_family"],
        "params": best_xgb_final["params"],
        "test_auc": best_xgb_final["test_cls_metrics"]["auc"],
        "test_f1": best_xgb_final["test_cls_metrics"]["f1"],
        "test_precision": best_xgb_final["test_cls_metrics"]["precision"],
        "test_recall": best_xgb_final["test_cls_metrics"]["recall"],
        "test_accuracy": best_xgb_final["test_cls_metrics"]["accuracy"],
        "test_mae": best_xgb_final["test_reg_metrics"]["mae"],
        "test_rmse": best_xgb_final["test_reg_metrics"]["rmse"],
    },
    {
        "model_family": best_lstm_final["model_family"],
        "params": best_lstm_final["params"],
        "test_auc": best_lstm_final["test_cls_metrics"]["auc"],
        "test_f1": best_lstm_final["test_cls_metrics"]["f1"],
        "test_precision": best_lstm_final["test_cls_metrics"]["precision"],
        "test_recall": best_lstm_final["test_cls_metrics"]["recall"],
        "test_accuracy": best_lstm_final["test_cls_metrics"]["accuracy"],
        "test_mae": best_lstm_final["test_reg_metrics"]["mae"],
        "test_rmse": best_lstm_final["test_reg_metrics"]["rmse"],
    },
]).sort_values(
    by=["test_auc", "test_f1", "test_mae", "test_rmse"],
    ascending=[False, False, True, True]
).reset_index(drop=True)

print("\n================================================")
print("FINAL TEST RESULTS")
print("================================================")
print(final_test_results)

best_overall_model = final_test_results.iloc[0].to_dict()

print("\n================================================")
print("BEST OVERALL MODEL")
print("================================================")
print(best_overall_model)


# ============================================================
# 18. ROC CURVES
# ============================================================

plt.figure(figsize=(8, 6))

xgb_prob = best_xgb_final["test_pred_cls"]
lstm_prob = best_lstm_final["test_pred_cls"]
y_test_cls = np.asarray(best_xgb_final["y_test_cls"]).astype(int)

for name, probs in [
    ("xgb_full_propagation", xgb_prob),
    ("lstm_context_full", lstm_prob),
]:
    fpr, tpr, _ = roc_curve(y_test_cls, probs)
    auc_val = roc_auc_score(y_test_cls, probs)
    plt.plot(fpr, tpr, label=f"{name} (AUC={auc_val:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves: Final Tuned Models on Nov-Dec Test")
plt.legend()
plt.tight_layout()
plt.show()


# ============================================================
# 19. ACTUAL VS PREDICTED DELAY
# ============================================================

best_name = final_test_results.iloc[0]["model_family"]

if best_name == "xgb_full_propagation":
    y_true_reg = np.asarray(best_xgb_final["y_test_reg"])
    y_pred_reg = np.asarray(best_xgb_final["test_pred_reg"])
else:
    y_true_reg = np.asarray(best_lstm_final["y_test_reg"])
    y_pred_reg = np.asarray(best_lstm_final["test_pred_reg"])

sample_n = min(5000, len(y_true_reg))
idx = np.random.RandomState(42).choice(len(y_true_reg), size=sample_n, replace=False)

plt.figure(figsize=(8, 6))
plt.scatter(y_true_reg[idx], y_pred_reg[idx], alpha=0.3)
plt.xlabel("Actual Arrival Delay (minutes)")
plt.ylabel("Predicted Arrival Delay (minutes)")
plt.title(f"Actual vs Predicted Delay: {best_name}")
plt.tight_layout()
plt.show()


# ============================================================
# 20. FEATURE IMPORTANCE FOR BEST XGB
# ============================================================

importance_df = pd.DataFrame({
    "feature": best_xgb_final["features"],
    "importance": best_xgb_final["classifier"].feature_importances_
}).sort_values("importance", ascending=False).head(20)

plt.figure(figsize=(10, 7))
plt.barh(importance_df["feature"][::-1], importance_df["importance"][::-1])
plt.title("Top Feature Importances: Tuned xgb_full_propagation")
plt.tight_layout()
plt.show()


# ============================================================
# 21. SAVE RESULTS OBJECT
# ============================================================

tuned_model_results = {
    "cv_splits": [
        {
            "fold": s["fold"],
            "val_start": str(s["val_start"].date()),
            "val_end": str(s["val_end"].date()),
            "train_rows": s["train_df"].height,
            "val_rows": s["val_df"].height,
        }
        for s in cv_splits
    ],
    "xgb_cv_results": xgb_cv_results,
    "lstm_cv_results": lstm_cv_results,
    "best_xgb_row": best_xgb_row,
    "best_lstm_row": best_lstm_row,
    "best_xgb_final": best_xgb_final,
    "best_lstm_final": best_lstm_final,
    "final_test_results": final_test_results,
    "best_overall_model": best_overall_model,
}

print("\nSaved results to `tuned_model_results`")
timer_log("TOTAL PIPELINE TIME", GLOBAL_START)

# Cluade 

In [ ]:
# ================================================================
# Flight Delay — XGBoost vs LSTM
# Grid Search + Walk-Forward CV  |  Memory & Speed Optimized
#
# Key optimizations vs previous version
# ──────────────────────────────────────
# 1. SLIM COLLECT: only the ~30 columns actually used in modeling
#    are kept after .collect(). Drops all string/metadata columns
#    immediately, cutting the in-memory DataFrame by ~50%.
#
# 2. TYPED STORAGE: float32 (not float64) for all numeric feature
#    columns, halving numpy/pandas array footprints. Polars Int8/
#    Int16 used for flags and small integers.
#
# 3. ONE PANDAS CONVERSION, EVER: the full feature matrix is
#    converted to numpy float32 once in Section 4. Every fold split
#    is a boolean-mask row slice of that array — zero repeated
#    .to_pandas() calls inside loops.
#
# 4. MEDIAN CACHE: fill medians are computed once on the Jan–Oct
#    training window and reused across every fold and every combo,
#    eliminating redundant per-call median computation.
#
# 5. LSTM ARRAYS BUILT ONCE: the (N,3,F) tensor is allocated once
#    from the pre-filled pandas frame, then immediately freed.
#    Fold splits are row-index slices of that single array.
#
# 6. SCALER REUSE WITHIN GRID-SEARCH FOLD: Fold 1 arrays are
#    scaled once and shared across all combos in the grid search.
#
# 7. LARGER LSTM BATCH (512): roughly halves the number of gradient
#    steps per epoch with negligible accuracy impact at this scale.
#
# 8. EXPLICIT del + gc.collect() after each LSTM combo to release
#    TF graph memory that clear_session() alone sometimes misses.
#
# 9. XGB early_stopping_rounds replaces sweeping n_estimators —
#    avoids training redundant trees and removes one grid axis.
#
# Walk-forward CV strategy (temporal safety)
# ─────────────────────────────────────────
#   Fold 1  Jan–Jun train  │ Jul–Aug  val   ← grid-search fold
#   Fold 2  Jan–Aug train  │ Sep–Oct  val
#   Fold 3  Jan–Oct train  │ Nov–Dec  val   ← canonical holdout
#
# Grid search runs on Fold 1 only. Best params are then evaluated
# across all 3 folds to measure seasonal stability.
# ================================================================

from pathlib import Path
import sys, importlib, warnings, itertools, json, gc
warnings.filterwarnings("ignore")
import os, time

# ── Multi-core env vars MUST be set before importing numpy/tf ──
import multiprocessing
N_CORES = multiprocessing.cpu_count()
print(f"[SYSTEM] {N_CORES} CPU cores")
for _var in ["OMP_NUM_THREADS","MKL_NUM_THREADS","OPENBLAS_NUM_THREADS",
             "NUMEXPR_NUM_THREADS","TF_NUM_INTEROP_THREADS","TF_NUM_INTRAOP_THREADS"]:
    os.environ[_var] = str(N_CORES)

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score,
    accuracy_score, mean_absolute_error, mean_squared_error, roc_curve,
)
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier, XGBRegressor
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

print(f"[TF] inter_op={tf.config.threading.get_inter_op_parallelism_threads()} "
      f"intra_op={tf.config.threading.get_intra_op_parallelism_threads()}")


# ================================================================
# GLOBAL CONFIG  ← tune these to trade speed for coverage
# ================================================================

USE_GPU        = True   # False → forces CPU for both XGB and TF
FAST_MODE      = False  # True  → minimal grids for a smoke-test run

# Walk-forward folds: (label, train_end_exclusive, val_end_exclusive)
# Training always starts from TRAIN_START (expanding window).
TRAIN_START = "2019-01-01"
CV_FOLDS = [
    ("Fold1 Jul–Aug",  "2019-07-01", "2019-09-01"),  # grid-search fold
    ("Fold2 Sep–Oct",  "2019-09-01", "2019-11-01"),
    ("Fold3 Nov–Dec",  "2019-11-01", "2020-01-01"),  # canonical holdout
]

LSTM_EPOCHS     = 15    # hard cap — EarlyStopping (patience=3) cuts short
LSTM_VAL_SPLIT  = 0.15
LSTM_BATCH_SIZE = 512   # up from 256; faster per-epoch, negligible accuracy cost
LSTM_LR         = 1e-3

XGB_N_EST      = 500    # upper bound; early stopping finds true optimum
XGB_EARLY_STOP = 30     # rounds without improvement on eval slice

# ── Hyperparameter grids ──────────────────────────────────────
# XGB: 24 combos. n_estimators NOT swept — early stopping handles it.
XGB_PARAM_GRID = {
    "max_depth":        [4, 6, 8],
    "learning_rate":    [0.03, 0.07],
    "subsample":        [0.75, 0.90],
    "colsample_bytree": [0.75, 0.90],
}
# LSTM: 12 combos. lr and batch_size fixed; width + dropout dominate.
LSTM_PARAM_GRID = {
    "units1":  [64, 128],
    "units2":  [32, 64],
    "dropout": [0.1, 0.2, 0.3],
}

if FAST_MODE:
    XGB_PARAM_GRID  = {"max_depth": [4, 6], "learning_rate": [0.07],
                       "subsample": [0.80], "colsample_bytree": [0.80]}
    LSTM_PARAM_GRID = {"units1": [64], "units2": [32], "dropout": [0.2]}


# ================================================================
# HELPERS
# ================================================================

GLOBAL_START = time.perf_counter()

def timer_log(label: str, t0: float) -> None:
    print(f"[TIMER] {label}: {time.perf_counter() - t0:.1f}s")

def section(title: str) -> None:
    print(f"\n{'='*64}\n  {title}\n{'='*64}")


# ── TF device setup ───────────────────────────────────────────
try:
    if USE_GPU:
        gpus = tf.config.list_physical_devices("GPU")
        if gpus:
            for g in gpus:
                tf.config.experimental.set_memory_growth(g, True)
            print(f"[TF] GPU mode — {len(gpus)} device(s)")
        else:
            print("[TF] No GPU found — using CPU.")
    else:
        tf.config.set_visible_devices([], "GPU")
        print("[TF] CPU-only mode.")
except RuntimeError as e:
    print(f"[TF WARNING] {e}")


# ================================================================
# 1. LOAD DATA
# ================================================================
section("1. LOAD DATA")
t0 = time.perf_counter()

pl.Config.set_tbl_rows(20); pl.Config.set_tbl_cols(50)

current = Path.cwd()
while current.name != "shared-notebooks":
    if current.parent == current:
        raise RuntimeError("Could not find shared-notebooks root")
    current = current.parent
utils_path = current / "common_utils" / "python"
if str(utils_path) not in sys.path:
    sys.path.append(str(utils_path))

import load_flight_data; importlib.reload(load_flight_data)
lf = load_flight_data.load_flight_data(file_name="flights_canonical_2019.parquet")
timer_log("Load data", t0)


# ================================================================
# 2. FEATURE ENGINEERING  (lazy — no .collect() until end)
# ================================================================
section("2. FEATURE ENGINEERING")
t0 = time.perf_counter()

# Only the columns we actually need downstream — avoids collecting
# string/metadata cols that inflate RAM without being used.
RAW_KEEP = [
    "flight_id", "tail_number", "route_key", "origin", "dest",
    "distance", "flight_date",
    "dep_hour_local", "dep_weekday_local", "dep_month_local",
    "dep_ts_actual_utc", "arr_ts_actual_utc",
    "dep_temp_c", "dep_wind_speed_m_s", "dep_wind_dir_deg", "dep_ceiling_height_m",
    "arr_temp_c", "arr_wind_speed_m_s", "arr_wind_dir_deg", "arr_ceiling_height_m",
    "prev_arr_delay", "prev_dep_delay", "prev_arr_del15", "prev_dep_del15",
    "dep_delay", "dep_del15", "arr_delay", "arr_del15",
    "turnaround_minutes", "rotation_continuity_flag",
    "aircraft_leg_number_day",
    "cum_dep_delay_aircraft_day", "cum_arr_delay_aircraft_day",
    "is_cancelled", "is_diverted", "crs_elapsed_time",
]

US_HOLIDAYS_2019 = [
    "2019-01-01","2019-01-21","2019-02-18","2019-05-27",
    "2019-07-04","2019-09-02","2019-10-14","2019-11-11",
    "2019-11-28","2019-12-25",
]

lf_base = (
    lf.select(RAW_KEEP)
    .filter(
        (pl.col("is_cancelled") == 0) &
        (pl.col("is_diverted")  == 0) &
        pl.col("arr_del15").is_not_null() &
        pl.col("tail_number").is_not_null() &
        pl.col("dep_ts_actual_utc").is_not_null() &
        pl.col("arr_ts_actual_utc").is_not_null()
    )
)

lf_eng = lf_base.with_columns([
    # Time bucket (Int8 — 6 values)
    pl.when(pl.col("dep_hour_local") < 6).then(pl.lit(1, dtype=pl.Int8))
      .when(pl.col("dep_hour_local") < 11).then(pl.lit(2, dtype=pl.Int8))
      .when(pl.col("dep_hour_local") < 14).then(pl.lit(3, dtype=pl.Int8))
      .when(pl.col("dep_hour_local") < 18).then(pl.lit(4, dtype=pl.Int8))
      .when(pl.col("dep_hour_local") < 21).then(pl.lit(5, dtype=pl.Int8))
      .otherwise(pl.lit(6, dtype=pl.Int8)).alias("dep_time_bucket"),
    # Binary flags (Int8)
    pl.col("dep_weekday_local").is_in([6,7]).cast(pl.Int8).alias("is_weekend"),
    pl.col("flight_date").cast(pl.Utf8).is_in(US_HOLIDAYS_2019).cast(pl.Int8).alias("is_holiday"),
    # Days to nearest holiday (Int16 — max 365)
    pl.min_horizontal([
        (pl.col("flight_date").cast(pl.Date) - pl.lit(h).str.strptime(pl.Date))
          .abs().dt.total_days()
        for h in US_HOLIDAYS_2019
    ]).cast(pl.Int16).alias("days_to_nearest_holiday"),
    # Traffic proxies (Int32)
    pl.len().over("route_key").cast(pl.Int32).alias("route_frequency"),
    pl.len().over("origin").cast(pl.Int32).alias("origin_flight_volume"),
    pl.len().over("dest").cast(pl.Int32).alias("dest_flight_volume"),
    # Propagation helpers
    (pl.col("prev_arr_delay") > 15).cast(pl.Int8).alias("prev_arr_delayed_flag"),
    (pl.col("prev_arr_delay") + pl.col("prev_dep_delay")).alias("prev_total_delay"),
    (pl.col("turnaround_minutes") < 60).cast(pl.Int8).alias("tight_turnaround_flag"),
    # Relative leg position (Float32)
    (
        pl.col("aircraft_leg_number_day").cast(pl.Float32) /
        pl.max("aircraft_leg_number_day").over("tail_number").cast(pl.Float32)
    ).alias("relative_leg_position"),
])

# 2-hop chain via temporal shift within each tail_number group
lf_hops = (
    lf_eng
    .sort(["tail_number", "dep_ts_actual_utc"])
    .with_columns([
        pl.col("flight_id").shift(1).over("tail_number").alias("prev1_flight_id"),
        pl.col("arr_delay").shift(1).over("tail_number").alias("prev1_arr_delay"),
        pl.col("dep_delay").shift(1).over("tail_number").alias("prev1_dep_delay"),
        pl.col("arr_del15").shift(1).over("tail_number").alias("prev1_arr_del15"),
        pl.col("dep_del15").shift(1).over("tail_number").alias("prev1_dep_del15"),
        pl.col("flight_id").shift(2).over("tail_number").alias("prev2_flight_id"),
        pl.col("arr_delay").shift(2).over("tail_number").alias("prev2_arr_delay"),
        pl.col("dep_delay").shift(2).over("tail_number").alias("prev2_dep_delay"),
        pl.col("arr_del15").shift(2).over("tail_number").alias("prev2_arr_del15"),
        pl.col("dep_del15").shift(2).over("tail_number").alias("prev2_dep_del15"),
        (
            pl.col("dep_ts_actual_utc") -
            pl.col("arr_ts_actual_utc").shift(1).over("tail_number")
        ).dt.total_minutes().alias("prev1_turnaround_minutes"),
        (
            pl.col("dep_ts_actual_utc") -
            pl.col("arr_ts_actual_utc").shift(2).over("tail_number")
        ).dt.total_minutes().alias("time_since_prev2_arrival_minutes"),
    ])
    .filter(
        pl.col("prev1_flight_id").is_not_null() &
        pl.col("prev2_flight_id").is_not_null() &
        pl.col("prev1_turnaround_minutes").is_not_null() &
        pl.col("time_since_prev2_arrival_minutes").is_not_null() &
        pl.col("prev1_turnaround_minutes").is_between(0, 720) &
        pl.col("time_since_prev2_arrival_minutes").is_between(0, 1440)
    )
    .with_columns(pl.col("dep_ts_actual_utc").cast(pl.Datetime))
)

# ── SLIM COLLECT — only columns used in model matrices ────────
KEEP_COLS = [
    "dep_ts_actual_utc",
    "arr_del15", "arr_delay",                          # targets
    "distance", "dep_hour_local", "dep_weekday_local", "dep_month_local",
    "dep_time_bucket", "is_weekend", "is_holiday", "days_to_nearest_holiday",
    "crs_elapsed_time",
    "dep_temp_c", "dep_wind_speed_m_s", "dep_wind_dir_deg", "dep_ceiling_height_m",
    "arr_temp_c", "arr_wind_speed_m_s", "arr_wind_dir_deg", "arr_ceiling_height_m",
    "route_frequency", "origin_flight_volume", "dest_flight_volume",
    "tight_turnaround_flag", "relative_leg_position",
    "cum_dep_delay_aircraft_day", "cum_arr_delay_aircraft_day",
    "prev_arr_delay", "prev_dep_delay", "prev_arr_del15", "prev_dep_del15",
    "prev_arr_delayed_flag", "prev_total_delay",
    "turnaround_minutes", "rotation_continuity_flag", "aircraft_leg_number_day",
    "prev1_arr_delay", "prev1_dep_delay", "prev1_arr_del15", "prev1_dep_del15",
    "prev2_arr_delay", "prev2_dep_delay", "prev2_arr_del15", "prev2_dep_del15",
    "prev1_turnaround_minutes", "time_since_prev2_arrival_minutes",
]

flights = lf_hops.select(KEEP_COLS).sort("dep_ts_actual_utc").collect()
print(f"Rows: {flights.height:,}  Cols: {flights.width}  "
      f"~{flights.estimated_size('mb'):.0f} MB")
timer_log("Feature engineering + collect", t0)


# ================================================================
# 3. FEATURE LISTS
# ================================================================

XGB_FEATURES = [
    "distance", "dep_hour_local", "dep_weekday_local", "dep_month_local",
    "dep_time_bucket", "is_weekend", "is_holiday", "days_to_nearest_holiday",
    "crs_elapsed_time",
    "dep_temp_c", "dep_wind_speed_m_s", "dep_wind_dir_deg", "dep_ceiling_height_m",
    "arr_temp_c", "arr_wind_speed_m_s", "arr_wind_dir_deg", "arr_ceiling_height_m",
    "route_frequency", "origin_flight_volume", "dest_flight_volume",
    "prev_arr_delay", "prev_dep_delay", "prev_arr_del15", "prev_dep_del15",
    "prev_arr_delayed_flag", "prev_total_delay",
    "turnaround_minutes", "tight_turnaround_flag", "rotation_continuity_flag",
    "aircraft_leg_number_day", "relative_leg_position",
    "cum_dep_delay_aircraft_day", "cum_arr_delay_aircraft_day",
    "prev1_arr_delay", "prev1_dep_delay", "prev1_arr_del15", "prev1_dep_del15",
    "prev2_arr_delay", "prev2_dep_delay", "prev2_arr_del15", "prev2_dep_del15",
    "prev1_turnaround_minutes", "time_since_prev2_arrival_minutes",
]

# Features placed in the *current-flight* slot of the LSTM sequence
LSTM_CURRENT_COLS = [
    "distance", "dep_hour_local", "dep_weekday_local", "dep_month_local",
    "dep_time_bucket", "is_weekend", "is_holiday", "days_to_nearest_holiday",
    "crs_elapsed_time",
    "dep_temp_c", "dep_wind_speed_m_s", "dep_wind_dir_deg", "dep_ceiling_height_m",
    "arr_temp_c", "arr_wind_speed_m_s", "arr_wind_dir_deg", "arr_ceiling_height_m",
    "route_frequency", "origin_flight_volume", "dest_flight_volume",
    "tight_turnaround_flag", "relative_leg_position",
    "cum_dep_delay_aircraft_day", "cum_arr_delay_aircraft_day",
]

# Fixed axis-2 layout for the (N, 3, F) LSTM tensor
STEP_FEATURES = [
    "arr_delay_prev", "dep_delay_prev", "arr_del15_prev", "dep_del15_prev",
    "gap_minutes",
] + LSTM_CURRENT_COLS

SF_IDX       = {f: i for i, f in enumerate(STEP_FEATURES)}
N_STEP_FEAT  = len(STEP_FEATURES)

TARGET_CLASS = "arr_del15"
TARGET_REG   = "arr_delay"


# ================================================================
# 4. PRE-COMPUTE NUMPY ARRAYS  (once — fold splits are row slices)
# ================================================================
section("4. PRE-COMPUTE NUMPY ARRAYS")
t0 = time.perf_counter()

# Timestamps array for fold boundary lookups
ts_np = flights["dep_ts_actual_utc"].to_numpy().astype("datetime64[ns]")

def fold_masks(train_end: str, val_end: str,
               train_start: str = TRAIN_START) -> tuple[np.ndarray, np.ndarray]:
    """Boolean masks for train/val rows using pre-computed ts_np."""
    t_s = np.datetime64(train_start, "ns")
    t_e = np.datetime64(train_end,   "ns")
    v_e = np.datetime64(val_end,     "ns")
    return (ts_np >= t_s) & (ts_np < t_e), (ts_np >= t_e) & (ts_np < v_e)

# Fold 3 training mask (Jan–Oct) used for median computation
f3_tr_mask, _ = fold_masks(CV_FOLDS[-1][1], CV_FOLDS[-1][2])

# ── XGB matrix (N, F) float32 ─────────────────────────────────
xgb_pdf = flights.select(XGB_FEATURES).to_pandas()
XGB_MEDIANS = xgb_pdf.iloc[f3_tr_mask].median()        # computed once
xgb_pdf.fillna(XGB_MEDIANS, inplace=True)
X_XGB = xgb_pdf.to_numpy(dtype=np.float32)
del xgb_pdf; gc.collect()

# ── Targets ───────────────────────────────────────────────────
Y_CLS = flights[TARGET_CLASS].to_numpy().astype(np.int8)
Y_REG = flights[TARGET_REG].to_numpy().astype(np.float32)

# ── LSTM tensor (N, 3, F) float32 ─────────────────────────────
# One full pandas frame for median fill, then immediately freed.
print("  Building LSTM (N,3,F) tensor …", end=" ", flush=True)
_fl = flights.to_pandas()

LSTM_FILL_COLS = [
    "prev2_arr_delay","prev2_dep_delay","prev2_arr_del15","prev2_dep_del15",
    "prev1_arr_delay","prev1_dep_delay","prev1_arr_del15","prev1_dep_del15",
    "prev1_turnaround_minutes","time_since_prev2_arrival_minutes",
] + LSTM_CURRENT_COLS

_tr_pdf = _fl.iloc[f3_tr_mask]
LSTM_MEDIANS = {c: _tr_pdf[c].median() for c in LSTM_FILL_COLS if c in _fl.columns}
for c, m in LSTM_MEDIANS.items():
    _fl[c].fillna(m, inplace=True)

N = len(_fl)
X_LSTM = np.zeros((N, 3, N_STEP_FEAT), dtype=np.float32)

X_LSTM[:, 0, SF_IDX["arr_delay_prev"]] = _fl["prev2_arr_delay"].values
X_LSTM[:, 0, SF_IDX["dep_delay_prev"]] = _fl["prev2_dep_delay"].values
X_LSTM[:, 0, SF_IDX["arr_del15_prev"]] = _fl["prev2_arr_del15"].values
X_LSTM[:, 0, SF_IDX["dep_del15_prev"]] = _fl["prev2_dep_del15"].values
X_LSTM[:, 0, SF_IDX["gap_minutes"]]    = _fl["time_since_prev2_arrival_minutes"].values

X_LSTM[:, 1, SF_IDX["arr_delay_prev"]] = _fl["prev1_arr_delay"].values
X_LSTM[:, 1, SF_IDX["dep_delay_prev"]] = _fl["prev1_dep_delay"].values
X_LSTM[:, 1, SF_IDX["arr_del15_prev"]] = _fl["prev1_arr_del15"].values
X_LSTM[:, 1, SF_IDX["dep_del15_prev"]] = _fl["prev1_dep_del15"].values
X_LSTM[:, 1, SF_IDX["gap_minutes"]]    = _fl["prev1_turnaround_minutes"].values

for col in LSTM_CURRENT_COLS:
    X_LSTM[:, 2, SF_IDX[col]] = _fl[col].values

del _fl, _tr_pdf; gc.collect()
print(f"done  shape={X_LSTM.shape}  ~{X_LSTM.nbytes/1024**2:.0f} MB")
timer_log("Pre-compute arrays", t0)


# ================================================================
# 5. METRIC HELPERS
# ================================================================

def cls_metrics(y_true, y_prob, thr: float = 0.5) -> dict:
    yp = (y_prob >= thr).astype(int)
    return {
        "auc":       round(float(roc_auc_score(y_true, y_prob)), 5),
        "f1":        round(float(f1_score(y_true, yp, zero_division=0)), 5),
        "precision": round(float(precision_score(y_true, yp, zero_division=0)), 5),
        "recall":    round(float(recall_score(y_true, yp, zero_division=0)), 5),
        "accuracy":  round(float(accuracy_score(y_true, yp)), 5),
    }

def reg_metrics(y_true, y_pred) -> dict:
    return {
        "mae":  round(float(mean_absolute_error(y_true, y_pred)), 4),
        "rmse": round(float(np.sqrt(mean_squared_error(y_true, y_pred))), 4),
    }


# ================================================================
# 6. MODEL HELPERS
# ================================================================

XGB_DEVICE = "cuda" if USE_GPU else "cpu"

def xgb_fit_es(params: dict,
               X_tr: np.ndarray, y_tr: np.ndarray,
               X_es: np.ndarray, y_es: np.ndarray,
               X_va: np.ndarray,
               task: str = "cls"):
    """Fit XGB with early stopping; return (model, val_preds)."""
    common = dict(**params, n_estimators=XGB_N_EST,
                  early_stopping_rounds=XGB_EARLY_STOP,
                  n_jobs=-1, device=XGB_DEVICE, random_state=42)
    if task == "cls":
        m = XGBClassifier(**common, eval_metric="logloss")
        m.fit(X_tr, y_tr, eval_set=[(X_es, y_es)], verbose=False)
        return m, m.predict_proba(X_va)[:, 1]
    m = XGBRegressor(**common, eval_metric="rmse")
    m.fit(X_tr, y_tr, eval_set=[(X_es, y_es)], verbose=False)
    return m, m.predict(X_va)


def scale_lstm(X_tr: np.ndarray, X_va: np.ndarray):
    nt, ts, nf = X_tr.shape
    sc  = StandardScaler()
    Xts = sc.fit_transform(X_tr.reshape(-1, nf)).reshape(nt, ts, nf).astype(np.float32)
    Xvs = sc.transform(X_va.reshape(-1, nf)).reshape(X_va.shape[0], ts, nf).astype(np.float32)
    return Xts, Xvs, sc


def build_lstm_model(input_shape, units1: int, units2: int,
                     dropout: float, lr: float = LSTM_LR) -> tf.keras.Model:
    inp     = Input(shape=input_shape)
    x       = LSTM(units1, return_sequences=True)(inp)
    x       = Dropout(dropout)(x)
    x       = LSTM(units2)(x)
    x       = Dropout(dropout)(x)
    cls_out = Dense(1, activation="sigmoid", name="cls")(x)
    reg_out = Dense(1, activation="linear",  name="reg")(x)
    model   = Model(inputs=inp, outputs=[cls_out, reg_out])
    model.compile(
        optimizer = tf.keras.optimizers.Adam(learning_rate=lr),
        loss      = {"cls": "binary_crossentropy", "reg": "mse"},
        metrics   = {"cls": [tf.keras.metrics.AUC(name="auc")], "reg": ["mae"]},
    )
    return model


def lstm_fit(params: dict,
             X_tr_s: np.ndarray, y_tr_c: np.ndarray, y_tr_r: np.ndarray,
             X_va_s: np.ndarray):
    """Train one LSTM combo; caller must clear_session() + gc before."""
    model = build_lstm_model((X_tr_s.shape[1], X_tr_s.shape[2]), **params)
    model.fit(
        X_tr_s,
        {"cls": y_tr_c.astype(np.float32), "reg": y_tr_r.astype(np.float32)},
        validation_split = LSTM_VAL_SPLIT,
        epochs           = LSTM_EPOCHS,
        batch_size       = LSTM_BATCH_SIZE,
        callbacks        = [EarlyStopping(monitor="val_loss", patience=3,
                                          restore_best_weights=True)],
        verbose=0,
    )
    pc, pr = model.predict(X_va_s, verbose=0)
    return model, pc.ravel(), pr.ravel()


def es_split(tr_mask: np.ndarray, es_frac: float = 0.10):
    """Split training indices into fit/early-stop slices (temporal order)."""
    idx   = np.where(tr_mask)[0]
    n_es  = max(1000, int(len(idx) * es_frac))
    return idx[:-n_es], idx[-n_es:]


# ================================================================
# 7. XGB GRID SEARCH  (Fold 1 only)
# ================================================================
section("7. XGB GRID SEARCH  (Fold 1)")
t0 = time.perf_counter()

def xgb_grid_search() -> tuple[dict, pd.DataFrame]:
    label, train_end, val_end = CV_FOLDS[0]
    tr_mask, va_mask = fold_masks(train_end, val_end)
    fit_idx, es_idx  = es_split(tr_mask)

    keys   = list(XGB_PARAM_GRID.keys())
    combos = list(itertools.product(*[XGB_PARAM_GRID[k] for k in keys]))
    print(f"  '{label}'  train={tr_mask.sum():,}  val={va_mask.sum():,}  "
          f"combos={len(combos)}\n")

    rows = []; best_auc = -np.inf; best_p = {}

    for i, vals in enumerate(combos, 1):
        params = dict(zip(keys, vals)); tc = time.perf_counter()

        clf, y_prob = xgb_fit_es(
            params,
            X_XGB[fit_idx], Y_CLS[fit_idx],
            X_XGB[es_idx],  Y_CLS[es_idx],
            X_XGB[va_mask], "cls",
        )
        reg, y_pred = xgb_fit_es(
            params,
            X_XGB[fit_idx], Y_REG[fit_idx],
            X_XGB[es_idx],  Y_REG[es_idx],
            X_XGB[va_mask], "reg",
        )
        cm = cls_metrics(Y_CLS[va_mask], y_prob)
        rm = reg_metrics(Y_REG[va_mask], y_pred)

        row = {"combo": i}; row.update(params)
        row.update({f"cls_{k}": v for k, v in cm.items()})
        row.update({f"reg_{k}": v for k, v in rm.items()})
        rows.append(row)

        print(f"  [{i:>2}/{len(combos)}] AUC={cm['auc']:.4f}  MAE={rm['mae']:.2f}  "
              f"trees={clf.best_iteration}  {params}  ({time.perf_counter()-tc:.1f}s)")

        if cm["auc"] > best_auc:
            best_auc = cm["auc"]; best_p = params.copy()

    df = pd.DataFrame(rows).sort_values("cls_auc", ascending=False).reset_index(drop=True)
    print(f"\n  ✓ XGB best AUC={best_auc:.4f}  params={best_p}")
    return best_p, df

xgb_best_params, xgb_gs_df = xgb_grid_search()
timer_log("XGB grid search", t0)


# ================================================================
# 8. LSTM GRID SEARCH  (Fold 1 only)
# ================================================================
section("8. LSTM GRID SEARCH  (Fold 1)")
t0 = time.perf_counter()

def lstm_grid_search() -> tuple[dict, pd.DataFrame]:
    label, train_end, val_end = CV_FOLDS[0]
    tr_mask, va_mask = fold_masks(train_end, val_end)

    # Scale once — reused across all combos in this fold
    X_tr_s, X_va_s, _ = scale_lstm(X_LSTM[tr_mask], X_LSTM[va_mask])

    keys   = list(LSTM_PARAM_GRID.keys())
    combos = list(itertools.product(*[LSTM_PARAM_GRID[k] for k in keys]))
    print(f"  '{label}'  train={tr_mask.sum():,}  val={va_mask.sum():,}  "
          f"combos={len(combos)}\n")

    rows = []; best_auc = -np.inf; best_p = {}

    for i, vals in enumerate(combos, 1):
        params = dict(zip(keys, vals)); tc = time.perf_counter()

        tf.keras.backend.clear_session(); gc.collect()
        _, pc, pr = lstm_fit(params, X_tr_s, Y_CLS[tr_mask], Y_REG[tr_mask], X_va_s)

        cm = cls_metrics(Y_CLS[va_mask], pc)
        rm = reg_metrics(Y_REG[va_mask], pr)

        row = {"combo": i}; row.update(params)
        row.update({f"cls_{k}": v for k, v in cm.items()})
        row.update({f"reg_{k}": v for k, v in rm.items()})
        rows.append(row)

        print(f"  [{i:>2}/{len(combos)}] AUC={cm['auc']:.4f}  MAE={rm['mae']:.2f}  "
              f"{params}  ({time.perf_counter()-tc:.1f}s)")

        if cm["auc"] > best_auc:
            best_auc = cm["auc"]; best_p = params.copy()

    tf.keras.backend.clear_session(); gc.collect()
    df = pd.DataFrame(rows).sort_values("cls_auc", ascending=False).reset_index(drop=True)
    print(f"\n  ✓ LSTM best AUC={best_auc:.4f}  params={best_p}")
    return best_p, df

lstm_best_params, lstm_gs_df = lstm_grid_search()
timer_log("LSTM grid search", t0)


# ================================================================
# 9. WALK-FORWARD CV  (best params, all 3 folds)
# ================================================================
section("9. WALK-FORWARD CV")

def run_cv(model_type: str, params: dict) -> pd.DataFrame:
    rows = []
    for label, train_end, val_end in CV_FOLDS:
        tf_m = time.perf_counter()
        tr_mask, va_mask = fold_masks(train_end, val_end)

        if model_type == "xgb":
            fit_idx, es_idx = es_split(tr_mask)
            _, y_prob = xgb_fit_es(params,
                X_XGB[fit_idx], Y_CLS[fit_idx], X_XGB[es_idx], Y_CLS[es_idx],
                X_XGB[va_mask], "cls")
            _, y_pred = xgb_fit_es(params,
                X_XGB[fit_idx], Y_REG[fit_idx], X_XGB[es_idx], Y_REG[es_idx],
                X_XGB[va_mask], "reg")
        else:
            X_tr_s, X_va_s, _ = scale_lstm(X_LSTM[tr_mask], X_LSTM[va_mask])
            tf.keras.backend.clear_session(); gc.collect()
            _, y_prob, y_pred = lstm_fit(
                params, X_tr_s, Y_CLS[tr_mask], Y_REG[tr_mask], X_va_s)
            tf.keras.backend.clear_session(); gc.collect()

        row = {"fold": label, "train_rows": int(tr_mask.sum()),
               "val_rows": int(va_mask.sum())}
        row.update({f"cls_{k}": v for k, v in cls_metrics(Y_CLS[va_mask], y_prob).items()})
        row.update({f"reg_{k}": v for k, v in reg_metrics(Y_REG[va_mask], y_pred).items()})
        rows.append(row)
        timer_log(f"  {model_type.upper()} {label}", tf_m)

    return pd.DataFrame(rows)

print("\n--- XGBoost CV ---")
t0 = time.perf_counter()
xgb_cv_df = run_cv("xgb", xgb_best_params)
timer_log("XGB CV total", t0)

print("\n--- LSTM CV ---")
t0 = time.perf_counter()
lstm_cv_df = run_cv("lstm", lstm_best_params)
timer_log("LSTM CV total", t0)

SHOW = ["fold","train_rows","val_rows","cls_auc","cls_f1","reg_mae","reg_rmse"]
print("\n── XGB CV ──"); print(xgb_cv_df[SHOW].to_string(index=False))
print("\n── LSTM CV ──"); print(lstm_cv_df[SHOW].to_string(index=False))


# ================================================================
# 10. FINAL MODELS  (Jan–Oct → Nov–Dec holdout)
# ================================================================
section("10. FINAL MODELS")

tr_mask_f, va_mask_f = fold_masks(CV_FOLDS[-1][1], CV_FOLDS[-1][2])
fit_idx_f, es_idx_f  = es_split(tr_mask_f)
print(f"  train={tr_mask_f.sum():,}  test={va_mask_f.sum():,}")

# XGB
t0 = time.perf_counter()
final_xgb_clf, xgb_prob_f = xgb_fit_es(
    xgb_best_params,
    X_XGB[fit_idx_f], Y_CLS[fit_idx_f],
    X_XGB[es_idx_f],  Y_CLS[es_idx_f],
    X_XGB[va_mask_f], "cls",
)
final_xgb_reg, xgb_pred_f = xgb_fit_es(
    xgb_best_params,
    X_XGB[fit_idx_f], Y_REG[fit_idx_f],
    X_XGB[es_idx_f],  Y_REG[es_idx_f],
    X_XGB[va_mask_f], "reg",
)
xgb_final_cls = cls_metrics(Y_CLS[va_mask_f], xgb_prob_f)
xgb_final_reg = reg_metrics(Y_REG[va_mask_f], xgb_pred_f)
timer_log("XGB final", t0)

# LSTM
t0 = time.perf_counter()
X_tr_s_f, X_va_s_f, _ = scale_lstm(X_LSTM[tr_mask_f], X_LSTM[va_mask_f])
tf.keras.backend.clear_session(); gc.collect()
final_lstm, lstm_prob_f, lstm_pred_f = lstm_fit(
    lstm_best_params, X_tr_s_f, Y_CLS[tr_mask_f], Y_REG[tr_mask_f], X_va_s_f)
lstm_final_cls = cls_metrics(Y_CLS[va_mask_f], lstm_prob_f)
lstm_final_reg = reg_metrics(Y_REG[va_mask_f], lstm_pred_f)
timer_log("LSTM final", t0)

comparison_df = pd.DataFrame([
    {"model": "XGB  (tuned)",
     **{f"cls_{k}": v for k, v in xgb_final_cls.items()},
     **{f"reg_{k}": v for k, v in xgb_final_reg.items()}},
    {"model": "LSTM (tuned)",
     **{f"cls_{k}": v for k, v in lstm_final_cls.items()},
     **{f"reg_{k}": v for k, v in lstm_final_reg.items()}},
])
cv_summary = pd.DataFrame([
    {"model": m,
     "auc_mean": d["cls_auc"].mean(), "auc_std": d["cls_auc"].std(),
     "f1_mean":  d["cls_f1"].mean(),  "f1_std":  d["cls_f1"].std(),
     "mae_mean": d["reg_mae"].mean(), "mae_std": d["reg_mae"].std()}
    for m, d in [("XGB (tuned)", xgb_cv_df), ("LSTM (tuned)", lstm_cv_df)]
])

print("\n── Final Holdout (Nov–Dec) ─────────────────────────────")
print(comparison_df.to_string(index=False))
print("\n── CV Summary (mean ± std, 3 folds) ───────────────────")
print(cv_summary.round(4).to_string(index=False))
print("\n── Best Hyperparameters ────────────────────────────────")
print("XGB :", json.dumps(xgb_best_params))
print("LSTM:", json.dumps(lstm_best_params))


# ================================================================
# 11. VISUALIZATIONS
# ================================================================
section("11. VISUALIZATIONS")

# ── 11a. Grid search score distributions ─────────────────────
def plot_gs_dist(gs_df, label):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(f"Grid Search Distributions — {label}", fontsize=12)
    for ax, col, best_fn, xlabel in [
        (axes[0], "cls_auc", max, "AUC"),
        (axes[1], "reg_mae", min, "MAE (min)"),
    ]:
        ax.hist(gs_df[col], bins=max(4, len(gs_df)//2), edgecolor="black", alpha=0.8)
        ax.axvline(best_fn(gs_df[col]), color="crimson", ls="--", label="best")
        ax.set_xlabel(xlabel); ax.set_ylabel("Count"); ax.legend(fontsize=8)
    plt.tight_layout(); plt.show()

plot_gs_dist(xgb_gs_df,  "XGB full propagation (Fold 1)")
plot_gs_dist(lstm_gs_df, "LSTM context full (Fold 1)")


# ── 11b. Parameter sensitivity box-plots ─────────────────────
def plot_param_sens(gs_df, params, label):
    fig, axes = plt.subplots(1, len(params), figsize=(5*len(params), 4))
    if len(params) == 1: axes = [axes]
    fig.suptitle(f"Param Sensitivity (AUC) — {label}", fontsize=11)
    for ax, p in zip(axes, params):
        vals = sorted(gs_df[p].unique())
        ax.boxplot([gs_df[gs_df[p] == v]["cls_auc"].values for v in vals],
                   labels=[str(v) for v in vals])
        ax.set_xlabel(p); ax.set_ylabel("AUC")
    plt.tight_layout(); plt.show()

plot_param_sens(xgb_gs_df,  ["max_depth","learning_rate","subsample"], "XGB")
plot_param_sens(lstm_gs_df, ["units1","units2","dropout"],              "LSTM")


# ── 11c. Walk-forward CV bars + trend lines ───────────────────
def plot_cv_stability(xdf, ldf):
    folds = xdf["fold"].tolist(); x = np.arange(len(folds)); w = 0.35

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax, col, ylabel, title, fmt in [
        (axes[0], "cls_auc", "AUC",        "AUC across CV Folds",  ".3f"),
        (axes[1], "reg_mae", "MAE (min)",   "MAE across CV Folds",  ".2f"),
    ]:
        b1 = ax.bar(x-w/2, xdf[col], w, label="XGB",  alpha=0.8)
        b2 = ax.bar(x+w/2, ldf[col], w, label="LSTM", alpha=0.8)
        ax.set_xticks(x); ax.set_xticklabels(folds, rotation=20, ha="right", fontsize=8)
        ax.set_ylabel(ylabel); ax.set_title(title); ax.legend()
        for bar in list(b1)+list(b2):
            h = bar.get_height()
            ax.text(bar.get_x()+bar.get_width()/2, h*1.002,
                    f"{h:{fmt}}", ha="center", va="bottom", fontsize=7)
    plt.suptitle("Walk-Forward CV Stability", fontsize=13)
    plt.tight_layout(); plt.show()

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fl = [f.replace(" ","\n") for f in folds]
    for ax, col, ylabel, title in [
        (axes[0], "cls_auc", "AUC",       "AUC Trend"),
        (axes[1], "reg_mae", "MAE (min)", "MAE Trend"),
    ]:
        ax.plot(fl, xdf[col], marker="o", label="XGB",  linewidth=1.8)
        ax.plot(fl, ldf[col], marker="s", label="LSTM", linewidth=1.8)
        ax.set_ylabel(ylabel); ax.set_title(title); ax.legend()
        ax.tick_params(axis="x", labelsize=8)
    plt.suptitle("Seasonal Stability — Walk-Forward CV", fontsize=12)
    plt.tight_layout(); plt.show()

plot_cv_stability(xgb_cv_df, lstm_cv_df)


# ── 11d. ROC curves — final holdout ──────────────────────────
y_te_c = Y_CLS[va_mask_f]; y_te_r = Y_REG[va_mask_f]
plt.figure(figsize=(8, 6))
for lbl, yp in [("XGB (tuned)", xgb_prob_f), ("LSTM (tuned)", lstm_prob_f)]:
    fpr, tpr, _ = roc_curve(y_te_c, yp)
    plt.plot(fpr, tpr, label=f"{lbl}  AUC={roc_auc_score(y_te_c, yp):.4f}", lw=1.8)
plt.plot([0,1],[0,1],"k--",lw=1)
plt.xlabel("FPR"); plt.ylabel("TPR")
plt.title("ROC Curves — Final Holdout (Nov–Dec)")
plt.legend(); plt.tight_layout(); plt.show()


# ── 11e. Actual vs Predicted scatter ─────────────────────────
def plot_avp(y_true, y_pred, label, n=5_000):
    rng = np.random.RandomState(42)
    idx = rng.choice(len(y_true), size=min(n, len(y_true)), replace=False)
    yt, yp = y_true[idx], np.array(y_pred)[idx]
    plt.figure(figsize=(7, 6))
    plt.scatter(yt, yp, alpha=0.25, s=8)
    lo, hi = min(yt.min(), yp.min()), max(yt.max(), yp.max())
    plt.plot([lo,hi],[lo,hi],"r--",lw=1,label="perfect fit")
    plt.xlabel("Actual Delay (min)"); plt.ylabel("Predicted Delay (min)")
    plt.title(f"Actual vs Predicted — {label}")
    plt.legend(); plt.tight_layout(); plt.show()

plot_avp(y_te_r, xgb_pred_f, "XGB (tuned, Nov–Dec)")
plot_avp(y_te_r, lstm_pred_f, "LSTM (tuned, Nov–Dec)")


# ── 11f. XGBoost feature importance ──────────────────────────
imp_df = (
    pd.DataFrame({"feature": XGB_FEATURES,
                  "importance": final_xgb_clf.feature_importances_})
    .sort_values("importance", ascending=False).head(20)
)
plt.figure(figsize=(10, 7))
plt.barh(imp_df["feature"][::-1], imp_df["importance"][::-1])
plt.title("Top-20 Feature Importances — Final XGB (Nov–Dec holdout)")
plt.tight_layout(); plt.show()


# ================================================================
# 12. SUMMARY TABLES
# ================================================================
section("12. SUMMARY")

xgb_cols  = ["combo"]+list(XGB_PARAM_GRID.keys()) +["cls_auc","cls_f1","reg_mae","reg_rmse"]
lstm_cols = ["combo"]+list(LSTM_PARAM_GRID.keys())+["cls_auc","cls_f1","reg_mae","reg_rmse"]

print("\n── XGB  Top Combos (Fold 1) ────────────────────────────")
print(xgb_gs_df.head(10)[xgb_cols].to_string(index=False))
print("\n── LSTM Top Combos (Fold 1) ────────────────────────────")
print(lstm_gs_df.head(10)[lstm_cols].to_string(index=False))
print("\n── XGB  CV Detail ──────────────────────────────────────")
print(xgb_cv_df[SHOW].to_string(index=False))
print("\n── LSTM CV Detail ──────────────────────────────────────")
print(lstm_cv_df[SHOW].to_string(index=False))
print("\n── CV Summary (mean ± std) ─────────────────────────────")
print(cv_summary.round(4).to_string(index=False))
print("\n── Final Holdout (Nov–Dec) ─────────────────────────────")
print(comparison_df.to_string(index=False))

print()
timer_log("TOTAL PIPELINE TIME", GLOBAL_START)